In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import cv2
import os

IMAGE_SIZE = 64
BOX_SIZE = 300

dataset_classes = sorted(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
			'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
			'del', 'nothing', 'space'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
class CNN(nn.Module):
        def __init__(self, num_classes):
                super().__init__()
                self.POOL_SIZE = 2
                self.L1_OUT_CHANNEL  = 32
                self.L1_KERNEL_SIZE  = 3
                self.L2_OUT_CHANNEL  = 64
                self.L2_KERNEL_SIZE  = 3
                self.L3_OUT_CHANNEL  = 128
                self.L3_KERNEL_SIZE  = 3
                self.DROPOUT_P = 0.5
                self.L1_NEURON = 512
                self.FINAL_DIM = IMAGE_SIZE // 8 # x Layers --> 2 ** x
                
                
                self.pool = nn.MaxPool2d(
                        kernel_size = self.POOL_SIZE,
                        stride = self.POOL_SIZE
                )
                self.relu = nn.ReLU()
                
                # Layer 1
                
                self.conv1 = nn.Conv2d(
                        in_channels = 3,
                        out_channels = self.L1_OUT_CHANNEL,
                        kernel_size =self. L1_KERNEL_SIZE,
                        padding = (self.L1_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm1 = nn.BatchNorm2d(self.L1_OUT_CHANNEL)
                
                # Layer  2
                
                self.conv2 = nn.Conv2d(
                        in_channels = self.L1_OUT_CHANNEL,
                        out_channels = self.L2_OUT_CHANNEL,
                        kernel_size = self.L2_KERNEL_SIZE,
                        padding = (self.L2_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm2 = nn.BatchNorm2d(self.L2_OUT_CHANNEL)
                
                # Layer  3
                self.conv3 = nn.Conv2d(
                        in_channels = self.L2_OUT_CHANNEL,
                        out_channels = self.L3_OUT_CHANNEL,
                        kernel_size = self.L3_KERNEL_SIZE,
                        padding = (self.L3_KERNEL_SIZE - 1) // 2
                )
                self.batch_norm3 = nn.BatchNorm2d(self.L3_OUT_CHANNEL)
                
                
                # Fully Connected (Dense) Layers
                self.dropout = nn.Dropout(self.DROPOUT_P)
                # After three MaxPools of 2x2, a 64x64 image becomes 8x8. 
                # So the flattened size is 128 channels * FINAL_DIM * FINAL_DIM
                
                self.fully_connected1 = nn.Linear(self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM, self.L1_NEURON)
                self.fully_connected2 = nn.Linear(self.L1_NEURON, num_classes)
                
        def forward(self, x: torch.Tensor):
                # L1
                x = self.pool(self.relu(self.batch_norm1(self.conv1(x))))
                
                # L2
                x = self.pool(self.relu(self.batch_norm2(self.conv2(x))))
                
                # L3
                x = self.pool(self.relu(self.batch_norm3(self.conv3(x))))
                
                x = x.view(-1, self.L3_OUT_CHANNEL * self.FINAL_DIM * self.FINAL_DIM)
                
                x = self.dropout(self.relu(self.fully_connected1(x)))
                x = self.fully_connected2(x)
                
                return x

In [12]:
model = CNN(num_classes=29).to(device)
model.load_state_dict(torch.load('my_sign_language_model.pth'))
model.eval()

inference_transforms = transforms.Compose([
    transforms.ToPILImage(),                 # ۱. تبدیل آرایه به PIL (امنتر)
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),                   # ۲. تبدیل به تنسور و مقیاس [0,1]
])

# inference_transforms = transforms.Compose([
#     transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
#     transforms.ToTensor(),
# ])


In [10]:
img_counter = 0          # counter for saved images

In [11]:

# Function to draw the box and prediction results on the frame
def draw_box_and_probs(frame, probs):
	h, w, _ = frame.shape

	# --- 1. Calculate the central square box coordinates ---
	center_x, center_y = w // 2, h // 2
	half = BOX_SIZE // 2
	x1 = max(0, center_x - half)
	y1 = max(0, center_y - half)
	x2 = min(w, center_x + half)
	y2 = min(h, center_y + half)

	# --- 2. Draw the box on the original frame (bright green) ---
	cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

	# --- 3. (Optional) Show a small thumbnail of the cropped region in the corner ---
	roi = frame[y1:y2, x1:x2]
	if roi.size > 0:
		roi_resized = cv2.resize(roi, (100, 100))
		frame[10:110, w-110:w-10] = roi_resized          # top‑right corner
		cv2.putText(frame, "Input", (w-110, 125),
			cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

	# --- 4. Show top‑3 classes on the image ---
	top_indices = np.argsort(probs)[0][::-1][:5]
	top_probs = probs[0][top_indices]

	y_offset = 30
	for i, (idx, prob) in enumerate(zip(top_indices, top_probs)):
		label = dataset_classes[idx]
		text = f"{label}: {prob*100:.1f}%"
		color = (0, 255, 0) if i == 0 else (0, 200, 200)
		cv2.putText(frame, text, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX,
			0.8, color, 2, cv2.LINE_AA)
		y_offset += 40

	# Small instruction for the user
	cv2.putText(frame, "Show your hand in the green border", (10, h-20),
			cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

	return frame, (x1, y1, x2, y2)


# Main webcam loop
cap = cv2.VideoCapture(0)
if not cap.isOpened():
	print("Error: Camera not found! (Try cap = cv2.VideoCapture(1) or 2)")
	exit()

print("Camera started. Show your hand inside the green box. Press 'q' to quit, 's' to save the current crop.")

# Create the output directory if it doesn't exist
save_dir = "real_data"
os.makedirs(save_dir, exist_ok=True)

saved_message_timer = 0  # timer to show "Saved!" on screen

while True:
	ret, frame = cap.read()
	if not ret:
		break

	h, w, _ = frame.shape

	# ---- 1. Crop the region inside the green box ----
	center_x, center_y = w // 2, h // 2
	half = BOX_SIZE // 2
	x1 = max(0, center_x - half)
	y1 = max(0, center_y - half)
	x2 = min(w, center_x + half)
	y2 = min(h, center_y + half)

	cropped = frame[y1:y2, x1:x2]   # the region of interest

	# If the box is fully inside the frame, process it
	if cropped.size > 0:
		# Convert BGR -> RGB
		rgb_cropped = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)

		# Apply transforms and run inference
		input_tensor = inference_transforms(rgb_cropped).unsqueeze(0).to(device)
		with torch.no_grad():
			output = model(input_tensor)
			probs = torch.softmax(output, dim=1).cpu().numpy()
	else:
		probs = np.zeros((1, len(dataset_classes)))   # fallback if crop fails

	# ---- 2. Draw box and predictions on the main frame ----
	frame, _ = draw_box_and_probs(frame, probs)

	# ---- 3. Check for key presses ----
	key = cv2.waitKey(1) & 0xFF

	# Save the current crop when 's' is pressed
	if key == ord('s'):
		if cropped.size > 0:
			filename = os.path.join(save_dir, f"pic_{img_counter}.png")
			cv2.imwrite(filename, cropped)
			print(f"Saved: {filename}")
			saved_message_timer = 30   # show "Saved!" for ~30 frames
			img_counter += 1
		else:
			print("Cannot save: crop is empty (box might be out of frame)")

	# Quit on 'q'
	if key == ord('q'):
		break

	# Show a temporary "Saved!" message if just saved
	if saved_message_timer > 0:
		cv2.putText(frame, "Saved!", (w//2 - 50, h//2),
			cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
		saved_message_timer -= 1

	# ---- 4. Display the final frame ----
	cv2.imshow("Recognizing sign language", frame)

cap.release()
cv2.destroyAllWindows()

Camera started. Show your hand inside the green box. Press 'q' to quit, 's' to save the current crop.
